<h1 align="center">CoNECTOME Hackathon</h1>  
<h1 align="center">Allen Institute/Neural Dynamics</h1>  
<h4 align="center"> May 20, 2026 </h4>   
<h4 align="center"> Dynamic Routing</h4> 
    

---
This version of the notebook uses the `lazynwb` package to work with large NWB files in the cloud more efficiently than `pynwb`. 

It is designed to run without Code Ocean and without AWS S3 credentials.

---

**Goal:** This tutorial will cover data access, details of the experiment, exploration and scientific analysis.

  We will:
  1. Find & load data packaged as NWB files
  2. Explain the design of the Dynamic Routing task
  3. Calculate behavioral performance metrics
  4. Summarize the brain areas recorded
  5. Explore data around stimulus onset: neural activity, running speed, face motion

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2 

In [ ]:
import pathlib
import warnings

import altair as alt
import lazynwb
import matplotlib.pyplot as plt
import numpy as np
import polars as pl

alt.renderers.enable("png", ppi=150)

lazynwb.config.anon = True # so we don't need credentials to access the data on S3

_ = pl.Config.set_tbl_rows(12)
_ = pl.Config.set_tbl_width_chars(-1)
_ = pl.Config.set_fmt_str_lengths(100)

---

## File inventory
Includes metadata (from .json files in each session data asset)

In [ ]:
remote_paths = pl.read_json("zarr.json")["nwb_path"].sort().to_list()
remote_paths

### Opening a file
<img src="../images/nwb-file.png" width="300">

Read one NWB-Zarr file using `lazynwb`:


In [ ]:
selected_path = remote_paths[3]

print(selected_path)

nwb = lazynwb.LazyNWB(selected_path)
print(type(nwb))


Instantiating the `LazyNWB` object (**above**) is lazy and fast.

Displaying the object in a notebook (**below**) eagerly loads all tables, so it can take much longer!


In [ ]:
# nwb

## Epochs within a session

The `NWBFile` in `pynwb` makes individual data containers accessible via a mix of dot attributes (`file.attr`) and mapping keys (`file.attr[key]`). 

In `lazynwb`, data containers are specified by their internal path.

"epochs" are stored as a table of intervals (i.e. each row has a `start_time` and a `stop_time`):


In [ ]:
epochs = lazynwb.read_nwb(selected_path, "/intervals/epochs")
epochs

Visualized, we can see that each session contains multiple epochs:
- `DynamicRouting1` is the task, which lasts about an hour
- some epochs are repeated after the task

In [ ]:
epochs.plot.bar(
    x="start_time",
    x2="stop_time",
    color="script_name"
)

Each epoch may be sub-divided into smaller intervals, e.g. trials or blocks of trials.

The epochs table gives us the name of the other intervals tables that correspond to each epoch.

In [ ]:
epochs.select("script_name", "interval_names")

---
# `DynamicRouting1` task trials



## `trials` table

- accessible at the internal path `/intervals/trials`

- **each row is one trial in the task**

- column types:
    - events (`*_time` suffix) 
        - `float`, seconds relative to start of session
    - boolean metadata (`is_*` prefix)
        - for easy filtering of rows
    - other metadata (`stim_name`, `rewarded_modality`, `block_index`) 
        - `int`, `str`
- column descriptions packaged in the NWB file:

    | column | description |
    |--------|-------------|
    | `stim_name` | the stimulus presented; corresponds to a unique stimulus definition, randomized over trials |
    | `rewarded_modality` | indicates the name of the rewarded modality in each block; not an exact correspondence with the values in `stim_name` |
    | `stim_start_time` | onset of visual or auditory stimulus |
    | `is_response` | the subject licked one or more times during the response window |
    | `reward_time` | delivery time of water reward, for contingent and non-contingent rewards |
    | ... | .. |


In [ ]:
trials_df = lazynwb.read_nwb(selected_path, "/intervals/trials")

Let's "discover" the block structure of the task by visualizing how the `rewarded_modality` changes across blocks:

In [ ]:
trials_df.plot.point(
    x="rewarded_modality",
    y="block_index:N",          # :N says this a "nominal" (or categorical) variable, not quantitative
)


In [ ]:
(
    # stimulus identity in the first few trials of each block:
    trials_df.filter(pl.col("trial_index_in_block") < 15)
    .plot.point(
        x="trial_index_in_block",
        y="stim_name",
        row="block_index",
        color="is_rewarded",
        tooltip=["stim_name", "is_target"],
    )
    .properties(height=60)
)


## Timing within a trial

All times in the NWB file are relative to the start of the session (ie before the first epoch starts).

We can get the response time relative to the stimulus onset in each trial:

In [ ]:
trials_df = trials_df.with_columns(
    (pl.col("response_time") - pl.col("stim_start_time")).alias("response_time_relative_to_stim")
)

<img src="../images/trial-timing.png" width="800">

Relevant columns:
- `quiescent_start_time` and `quiescent_stop_time`
- `stim_start_time`
- `response_time` -  when the mouse licked (`NaN` if no lick)
- `response_time_relative_to_stim` -  response time in seconds from stimulus onset

In [ ]:
trials_df.plot.tick(
    x="response_time_relative_to_stim",
    y="stim_name",
)

## Behavioral performance

We can aggregate trial outcomes into blockwise performance metrics, e.g. "response rate in each block" using polars `group_by`.

Alternatively, the NWB file contains a pre-computed `performance` table.


In [ ]:
performance_df = lazynwb.read_nwb(selected_path, "/intervals/performance")
performance_df.schema


In [ ]:
performance_df.plot.line(
    x="block_index:N",
    y="aud_target_response_rate",
).properties(height=50)

---
# Neural data

The `units` table contains metadata about each recorded (putative) neuron.

It also contains list-like `spike_times` and `spike_amplitudes` columns, and may contain array-like `waveform_mean` and `waveform_sd` columns which are be very large compared to the scalar metadata.

Let's load the units table as a deferred-evaluation "lazy" dataframe:

In [ ]:
units_lf = lazynwb.scan_nwb(selected_path, "/units")
units_lf

For more info, see the lazy API guide in the Polars documentation https://docs.pola.rs/user-guide/concepts/lazy-api/

In [ ]:
units_lf.collect_schema()


## Brain areas recorded

In [ ]:
ccf_lf = pl.scan_csv(
    "https://raw.githubusercontent.com/cortex-lab/allenCCF/refs/heads/master/structure_tree_safe_2017.csv"
).select("name", "acronym")

(
    units_lf
    .group_by("structure")
    .agg(
        pl.len().alias("n units"),
        pl.col("is_qc_pass").sum().alias("n passing qc"),
    )
    .join(ccf_lf, left_on="structure", right_on="acronym")
    .sort("structure")
    .select(
        pl.col("name").alias("description"),
        pl.col("structure").alias("structure (acronym)"),
        "n units",
        "n passing qc",
    )
).collect()


Let's visualize the location of all the recording sites that picked up units (excluding units labeled as "noise"):

In [ ]:
import lazynwb_utils

with alt.renderers.enable('default'):
    display(lazynwb_utils.plot_ccf_coordinates(units_lf))


## Single-unit exploration


### Metadata for a single unit

In [ ]:
UNIT_ID = "742903_2024-10-22_E-775"

(
    units_lf
    .filter(pl.col("unit_id") == UNIT_ID)
    .select(
        "structure",
        "location",
        "electrode_group_name",
        "is_qc_pass",
        "activity_drift",
    )
).collect()


`is_qc_pass` allows quick filtering for units whose activity is stable over the task, and that do not appear to be noise:
- `activity_drift < 0.1`
- `amplitude_cutoff <= 0.1`
-  `presence_ratio > 0.7`
- `decoder_label != "noise"`

### Windowed spike times
<img src="../images/spikes.png" width="800">

In [ ]:
spike_times = (
    lazynwb.scan_nwb(selected_path, "/units")
    .filter(pl.col("unit_id") == UNIT_ID)
    .select("spike_times")
    .collect()["spike_times"][0]
    .to_numpy()
)
truncated_spike_times = pl.DataFrame({"spike_times": spike_times[:100]})

alt.Chart(data=truncated_spike_times).mark_tick().encode(x="spike_times")


In [ ]:
(
    lazynwb.scan_nwb(selected_path, "/units")
    .filter(pl.col("unit_id") == UNIT_ID)
    .select("obs_intervals")
    .collect()
)


### Stimulus-aligned response (PSTH)

In [ ]:
query = "is_vis_stim"

# time window around stimulus onset
xmin = -1.5 # seconds
xmax = 2.0  # seconds
bin_size = 0.001

# count spikes in each bin, summed across all trials
n_bins = int(round((xmax - xmin) / bin_size))
bins = xmin + np.arange(n_bins + 1) * bin_size
counts = np.zeros(n_bins)
for stim_start in trials_df.filter(query)["stim_start_time"]:
    counts += np.histogram(spike_times - stim_start, bins=bins)[0]

smooth_ms = 100
smooth_bins = int(round((smooth_ms / 1000) / bin_size))
kernel = np.ones(smooth_bins) / smooth_bins
rate_hz = counts / len(trials_df.filter(query)) / bin_size
psth_hz = np.convolve(rate_hz, kernel, mode="valid")

# center time of each valid smoothing window
bin_centers = bins[:-1] + bin_size / 2
t = bin_centers[:len(psth_hz)] + (smooth_bins - 1) * bin_size / 2

plt.plot(t, psth_hz, lw=1.2)
plt.axvline(0, color="gray", lw=0.8)
plt.xlabel("time from stim onset (s)")
plt.ylabel("spikes/s")
plt.gcf().set_size_inches(3, 3)
plt.title(f"{UNIT_ID}\n{query!r} trials")
plt.show()


## Activity by stimulus and block

In [ ]:
unit_id = UNIT_ID

# or use a custom units filter and get a random unit:
query = (
    (pl.col("is_qc_pass"))
    & (pl.col("structure") == "MOs")
    & (pl.col("firing_rate") > 5)
)
random_unit_id = (
    units_lf
    .filter(query)
    .select("unit_id")
    .collect()
    .sample(1)["unit_id"][0]
)
# unit_id = random_unit_id

fig = lazynwb_utils.plot_unit_raster_psth(
    unit_id=unit_id,
    nwb_or_units_df=units_lf,
    trials_df=trials_df,
    stim_names=("vis+", "aud+", "vis-", "aud-"),
    max_psth_spike_rate=60,
)
plt.show()



---
# Behavioral data

## Accessing timeseries containers


In [ ]:
running_speed = lazynwb.get_timeseries(
    selected_path,
    "/processing/behavior/running_speed",
    exact_path=True,
)
display(running_speed.timestamps, running_speed.data)


In [ ]:
sampling_rate_hz = 1 / np.median(np.diff(running_speed.timestamps[:1000]))
sampling_rate_hz 

In [ ]:
n_samples_to_plot = int(sampling_rate_hz * 60 * 4)
start_sample = int(sampling_rate_hz * 60 * 70)  # at 70 min the task is ongoing

t = running_speed.timestamps[start_sample:start_sample + n_samples_to_plot]
y = running_speed.data[start_sample:start_sample + n_samples_to_plot]

plt.plot(t, y, lw=0.5)
plt.axhline(0, color="gray", linestyle="dashed")
plt.xlabel(f"time ({running_speed.timestamps_unit})")
plt.ylabel(f"running speed ({running_speed.unit})")
plt.show()


## Running speed by outcome and block

In [ ]:
running_time_grid = np.arange(-1.5, 2, 0.05)

# filter trials to hits and correct rejects, excluding instructions and catch
running_trials = (
    trials_df.filter(
        (pl.col("is_vis_target") | pl.col("is_aud_target"))
        & (pl.col("is_hit") | pl.col("is_correct_reject"))
    )
    .with_columns(
        pl.when(pl.col("is_hit"))
        .then(pl.lit("hit"))
        .otherwise(pl.lit("correct reject"))
        .alias("outcome")
    )
)

# load running speed as numpy arrays
rs_times = np.array(running_speed.timestamps)
rs_data = np.array(running_speed.data)

# align running speed to each trial's stimulus onset
aligned_rows = []
for trial in running_trials.iter_rows(named=True):
    sample_times = trial["stim_start_time"] + running_time_grid
    if sample_times[0] < rs_times[0] or sample_times[-1] > rs_times[-1]:
        continue
    aligned_rows.append({
        "rewarded_modality": trial["rewarded_modality"],
        "outcome": trial["outcome"],
        "running_speed": np.interp(sample_times, rs_times, rs_data),
    })

fig, axes = plt.subplots(1, 2, figsize=(8, 3), sharex=True, sharey=True)
for col_idx, outcome in enumerate(["hit", "correct reject"]):
    ax = axes[col_idx]
    for modality in ["vis", "aud"]:
        traces_for_group = [
            row["running_speed"] for row in aligned_rows
            if row["outcome"] == outcome and row["rewarded_modality"] == modality
        ]
        if not traces_for_group:
            continue
        traces = np.vstack(traces_for_group)
        mean = traces.mean(axis=0)
        sem = traces.std(axis=0, ddof=1) / np.sqrt(len(traces))
        ax.plot(running_time_grid, mean, label=f"{modality} rewarded (n={len(traces)})")
        ax.fill_between(running_time_grid, mean - sem, mean + sem, alpha=0.3)
    ax.axvline(0, color="gray")
    ax.set_title(outcome)
    ax.set_xlabel("seconds from stimulus onset")
    if col_idx == 0:
        ax.set_ylabel(f"running speed ({running_speed.unit})")
    ax.legend()
plt.tight_layout()
plt.show()


## Facial features from `lightning pose (lp)` 

In [ ]:
#TODO: lazynwb can show the table schema here, but not the pynwb DynamicTable widget.
lazynwb.get_table_schema(
    selected_path,
    "/processing/behavior/lp_side_camera",
    exclude_internal_columns=True,
)

<img src="../images/nose_cam.png" style="max-width: 600px; height: auto;">

In [ ]:
keypoint = "ear_base_l_y"
# trials_df = lazynwb.read_nwb(remote_paths[3], "/intervals/trials")

# filter trials (same criteria as running speed)
pose_trials = (
    trials_df.filter(
        (pl.col("is_vis_target") | pl.col("is_aud_target"))
        & (pl.col("is_hit") | pl.col("is_correct_reject"))
    )
    .with_columns(
        pl.when(pl.col("is_hit"))
        .then(pl.lit("hit"))
        .otherwise(pl.lit("correct reject"))
        .alias("outcome")
    )
)

# extract keypoint data
lp_df = (
    lazynwb.scan_nwb(selected_path, "/processing/behavior/lp_side_camera")
    .filter(pl.col(f"{keypoint[:-2]}_likelihood") > 0.99)  # only keep samples where the keypoint was confidently detected
    .select("timestamps", keypoint)
    .filter(pl.col("timestamps").is_finite(), pl.col(keypoint).is_finite())
    .collect()
)

pose_times = lp_df["timestamps"]
pose_values = lp_df[keypoint]

# align keypoint to each trial's stimulus onset
xmin = -1.5
pose_time_grid = np.arange(xmin, 2.0, 0.05)
aligned_rows = []
for trial in pose_trials.iter_rows(named=True):
    sample_times = trial["stim_start_time"] + pose_time_grid
    if sample_times[0] < pose_times[0] or sample_times[-1] > pose_times[-1]:
        continue
    values = np.interp(sample_times, pose_times, pose_values)
    aligned_rows.append({
        "rewarded_modality": trial["rewarded_modality"],
        "outcome": trial["outcome"],
        "values": values,
    })

# plot keypoint by outcome
fig, axes = plt.subplots(1, 2, figsize=(8, 3), sharex=True)
for col_idx, outcome in enumerate(["hit", "correct reject"]):
    ax = axes[col_idx]
    for modality in ["vis", "aud"]:
        traces_for_group = [
            row["values"] for row in aligned_rows
            if row["outcome"] == outcome and row["rewarded_modality"] == modality
        ]
        if not traces_for_group:
            continue
        traces = np.vstack(traces_for_group)
        mean = traces.mean(axis=0)
        sem = traces.std(axis=0, ddof=1) / np.sqrt(len(traces))
        ax.plot(pose_time_grid, mean, label=f"{modality} rewarded (n={len(traces)})")
        ax.fill_between(pose_time_grid, mean - sem, mean + sem, alpha=0.3)
    ax.axvline(0, color="gray")
    ax.set_title(outcome)
    ax.set_xlabel("seconds from stimulus onset")
    ax.set_ylabel("y position (pixels)")
    ax.legend(frameon=False)

plt.suptitle(keypoint)
plt.tight_layout()
plt.show()


---

## Multi-file operations

In [ ]:
(
    lazynwb.scan_nwb(
        remote_paths,
        "units",
    )
    .select("structure")
    .unique()
    .sort("structure")
).collect()